# 🔥 Fine-tuning Сидоровича (Unsloth QLoRA)
Обучает Qwen3-8B на датасете и экспортирует GGUF для LM Studio.


## 1. Установка зависимостей
Запусти один раз, потом можно пропускать.

In [1]:
import torch
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не найден'}")

CUDA доступна: False
Версия CUDA: None
GPU: не найден


In [2]:
# Для CUDA 12.x (большинство RTX 40xx)
!pip install "unsloth[cu121]" trl datasets transformers -q

# Если CUDA 11.8 — раскомментируй:
# !pip install "unsloth[cu118]" trl datasets transformers -q


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Настройки

In [3]:
DATASET_FILE  = "dataset.jsonl"
OUTPUT_DIR    = "./finetuned_model"
GGUF_DIR      = "./gguf_export"

# Модель (8B — оптимально для 8 ГБ VRAM с уменьшенными настройками)

MODEL_NAME    = "unsloth/Qwen3-8B-bnb-4bit"

MAX_SEQ_LENGTH = 1024   # уменьши до 512 если Out of Memory
LORA_R         = 16
LORA_ALPHA     = 16
BATCH_SIZE     = 1
GRAD_ACCUM     = 8      # эффективный batch = 8
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 3

print("Настройки загружены ✓")

Настройки загружены ✓


## 3. Загрузка модели
> Первый раз скачает ~4 ГБ, подожди.

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
print("Модель загружена ✓")

AssertionError: Torch not compiled with CUDA enabled

## 4. Применение LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA применена ✓")

## 5. Подготовка датасета

In [ ]:
import json
import os
from datasets import Dataset

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(f"{DATASET_FILE} не найден. Сначала запусти build_dataset.ipynb")

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f"Примеров в датасете: {len(raw_data)}")

if len(raw_data) < 10:
    print("⚠️ Мало примеров! Добавь больше фраз для лучшего качества.")

hf_dataset = Dataset.from_list(raw_data)

def format_to_chatml(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

hf_dataset = hf_dataset.map(
    format_to_chatml,
    batched=True,
    remove_columns=hf_dataset.column_names,
)

print("Датасет подготовлен ✓")
print("Пример:")
print(hf_dataset[0]["text"][:300])

## 6. Обучение

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=True,
        logging_steps=5,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
    ),
)

trainer_stats = trainer.train()
print(f"\n✅ Обучение завершено!")
print(f"Время: {trainer_stats.metrics['train_runtime']:.0f} сек")

## 7. Экспорт в GGUF для LM Studio

In [ ]:
import os
os.makedirs(GGUF_DIR, exist_ok=True)

print("Экспорт в GGUF Q4_K_M...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"\n✅ GGUF сохранён: {os.path.abspath(GGUF_DIR)}")
print("\nЧто дальше:")
print("1. Открой LM Studio")
print("2. My Models → Load model from disk")
print(f"3. Укажи папку: {os.path.abspath(GGUF_DIR)}")
print("4. Запусти сервер → бот готов!")

## 🧪 Быстрый тест модели (до экспорта)

In [ ]:
FastLanguageModel.for_inference(model)

TEST_MESSAGE = "Сидорович, что есть из снаряжения?"

inputs = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "Ты — Сидорович, торговец из S.T.A.L.K.E.R."},
        {"role": "user",   "content": TEST_MESSAGE},
    ],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=150,
    temperature=0.8,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(f"USER: {TEST_MESSAGE}")
print(f"BOT:  {response}")")